In [ ]:
# ===========================================================================
# PRAAT ACOUSTIC ANALYSIS - interactive driver (Phase 4)
#
# All extraction/plotting logic lives in src/; this notebook only calls it
# and stores results, matching notebooks/01 and 02's convention.
#
#   src/praat.py           per-feature-group extraction (F0, jitter, shimmer,
#                           HNR, formants, intensity, speech-rate/pause/voice-
#                           break proxies) + extract_praat_features_batch()
#   src/visualization.py   plot_praat_feature_comparison, build_praat_group_summary
#
# Features are extracted from the ORIGINAL audio, not the VAD-trimmed/
# zero-padded window the Deep/Acoustic pathways train on - jitter, shimmer,
# HNR, and formants are only meaningful on natural speech. See ROADMAP.md
# Phase 4 for the full plan and src/praat.py's module docstring for the
# speech-rate/pause-duration caveat (UA-Speech utterances are single
# isolated words, not continuous speech).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("Praat Acoustic Analysis")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))

In [ ]:
# STAGE 1 - Extract all 18 Praat features (F0 x3, jitter x3, shimmer x3, HNR,
# formants x3, intensity x2, speech-rate/pause/voice-breaks x3) for every
# utterance in the manifest. Cached to outputs/praat_features.csv - safe to
# re-run this cell, later runs load the cache instead of recomputing ~21k
# files (~25-30 minutes uncached).
from src.praat import extract_praat_features_batch

praat_features = extract_praat_features_batch(df_m6, cache_path=config.PRAAT_FEATURES_PATH)
praat_features.head()

In [ ]:
# STAGE 2 - Compare Healthy vs Very Low vs Low vs Mid vs High severity groups:
# box-plot grid for all 18 features, saved to outputs/figures/, plus a
# group-means +/- std table saved to outputs/metrics/.
from src.visualization import plot_praat_feature_comparison, build_praat_group_summary

figure_path = plot_praat_feature_comparison(praat_features, show=True)
group_summary = build_praat_group_summary(praat_features)

summary_path = config.METRICS_DIR / "praat_severity_group_summary.csv"
group_summary.to_csv(summary_path)

print_header("Phase 4 - Severity Group Comparison")
print_kv("Figure", figure_path)
print_kv("Group summary table", summary_path)
group_summary